# Feasibility-based bound tightening (FBBT)

`pounce` can run interval-arithmetic propagation through each
nonlinear constraint's expression DAG to tighten variable bounds
*before* the IPM starts. The classic illustration: a circle constraint
`x² + y² ≤ 1` with a wide declared box `x, y ∈ [-100, 100]` — FBBT
shrinks the box toward `[-1, 1]` for free.

On well-posed problems that tightening often doesn't change the
iteration count (we show this below). **Where FBBT earns its keep is
robustness.** When a model has loose bounds *and* constraints built
from restricted-domain functions (`log`, `sqrt`, division, fractional
powers), the un-tightened box lets the interior-point method evaluate
those functions where they are undefined — producing NaNs, restoration
failures, or a spurious "infeasible" verdict. FBBT removes those
regions before the solver ever steps into them. The two examples near
the end turn a **failure into a solve** and a **wrong answer into the
right one**.

It is off by default — issue
[#62](https://github.com/jkitchin/pounce/issues/62) defers the default
flip until benchmark evidence justifies it. This notebook shows the
mechanism first, then the cases where you'll want it on.

See the [FBBT reference](https://kitchingroup.cheme.cmu.edu/pounce/fbbt.html)
for the algorithm, operator support, and soundness guarantees.

In [1]:
import io
import contextlib
import pyomo_pounce  # noqa: F401 -- registers 'pounce' with SolverFactory
import pyomo.environ as pyo


## The mechanism: a problem with loose bounds

Two-variable optimization on a circle, with the box declared
100× larger than necessary:

$$
\min_{x, y}\; (x - 0.5)^2 + (y - 0.5)^2
\quad\text{s.t.}\quad x^2 + y^2 = 1,\; -100 \le x \le 100,\; -100 \le y \le 100.
$$

The optimum is the closest point on the unit circle to `(0.5, 0.5)` —
i.e. `(√2/2, √2/2) ≈ (0.707, 0.707)`. This problem is well-behaved
enough that the IPM solves it in 7 iterations no matter how loose the
box is, so treat it as a way to **see the mechanism** — watch the
`Σ|Δ|` field in the FBBT report show the box collapsing from `[-100,
100]²` into `[-1, 1]²` — not as proof of a speedup. The cases where
FBBT changes the *outcome* come right after.

In [2]:
def build_model():
    m = pyo.ConcreteModel()
    m.x = pyo.Var(bounds=(-100.0, 100.0), initialize=0.0)
    m.y = pyo.Var(bounds=(-100.0, 100.0), initialize=0.0)
    m.obj = pyo.Objective(expr=(m.x - 0.5)**2 + (m.y - 0.5)**2)
    m.circle = pyo.Constraint(expr=m.x**2 + m.y**2 == 1.0)
    return m


def solve(extra_options=None):
    """Solve once with `extra_options` merged onto a quiet baseline.
    Returns (n_iters, x*, y*, fbbt_report_line_or_None)."""
    m = build_model()
    solver = pyo.SolverFactory("pounce")
    solver.options["print_level"] = 5  # so we get the iter-count line
    solver.options["tol"] = 1e-8
    for k, v in (extra_options or {}).items():
        solver.options[k] = v
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        solver.solve(m, tee=True)
    log = buf.getvalue()
    iters = None
    fbbt_line = None
    for line in log.splitlines():
        if line.startswith("Number of Iterations"):
            iters = int(line.split(":")[1].strip())
        if line.startswith("Presolve FBBT:"):
            fbbt_line = line
    return iters, pyo.value(m.x), pyo.value(m.y), fbbt_line


## Run 1 — solve as-is (no presolve, no FBBT)

In [3]:
iters, x, y, fbbt = solve()
print(f"iters : {iters}")
print(f"x*    : {x:.6f}")
print(f"y*    : {y:.6f}")
print(f"|x|+|y| residual to circle: {abs(x*x + y*y - 1.0):.2e}")
print(f"FBBT report: {fbbt or 'none'}")

iters : 7
x*    : 0.707107
y*    : 0.707107
|x|+|y| residual to circle: 2.22e-16
FBBT report: none


## Run 2 — `presolve=yes` but no FBBT

Pounce's existing linear presolve doesn't do anything useful here:
the only constraint is nonlinear (it's a circle), so Phase 1's
linear bound-tightening has nothing to propagate.

In [4]:
iters, x, y, fbbt = solve({"presolve": "yes"})
print(f"iters : {iters}")
print(f"x*    : {x:.6f}, y* : {y:.6f}")
print(f"FBBT report: {fbbt or 'none'}")

iters : 7
x*    : 0.707107, y* : 0.707107
FBBT report: none


## Run 3 — turn on FBBT

`presolve=yes presolve_fbbt=yes`: now the circle constraint
propagates into the variable box. After one forward+reverse pass,
the box becomes a subset of `[-1, 1]²`.

In [5]:
iters, x, y, fbbt = solve({"presolve": "yes", "presolve_fbbt": "yes"})
print(f"iters : {iters}")
print(f"x*    : {x:.6f}, y* : {y:.6f}")
print(f"FBBT report: {fbbt}")

iters : 7
x*    : 0.707107, y* : 0.707107
FBBT report: Presolve FBBT: 2 sweeps, 2 variable tightenings (Σ|Δ|=1.980e2)


## Where FBBT changes the outcome

The circle was a wash on iterations — the honest, common case. Now the
cases that actually matter: **loose bounds combined with
restricted-domain functions.** Here FBBT is not an optimization, it is
what keeps the solver out of regions where the constraint functions are
undefined.

We'll use one helper that reports the exit status, iteration count, and
FBBT line, and only loads a solution when the solve actually
succeeds (so a solver failure prints a status instead of raising):

In [6]:
def solve_capture(build_model, extra_options=None):
    """Solve and return (status, iters, x*, fbbt_line). Reads the status from
    the log so a solver failure doesn't raise, and loads the primal point
    only when the solve actually succeeds."""
    m = build_model()
    solver = pyo.SolverFactory("pounce")
    solver.options["print_level"] = 5
    solver.options["tol"] = 1e-8
    for k, v in (extra_options or {}).items():
        solver.options[k] = v
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        results = solver.solve(m, tee=True, load_solutions=False)
    status, iters, fbbt = "?", None, None
    for line in buf.getvalue().splitlines():
        if line.startswith("EXIT"):
            status = line.split("EXIT:")[1].strip()
        if line.startswith("Number of Iterations"):
            iters = int(line.split(":")[1])
        if line.startswith("Presolve FBBT"):
            fbbt = line.strip()
    x = None
    if status.startswith("Optimal"):
        m.solutions.load_from(results)
        x = pyo.value(m.x)
    return status, iters, x, fbbt


def report(build_model):
    for label, opts in [("FBBT off", {}),
                        ("FBBT on", {"presolve": "yes", "presolve_fbbt": "yes"})]:
        status, iters, x, fbbt = solve_capture(build_model, opts)
        xstr = f"x*={x:.6f}" if x is not None else "(no solution returned)"
        print(f"  {label:8s}: {status:48s} iters={iters}  {xstr}")
        if fbbt:
            print(f"            {fbbt}")


# Example A — domain safety. log(x) is defined only for x > 0, but the box
# allows x <= 0 and we start at x = -500. Without FBBT the first evaluation
# of log(-500) is NaN and restoration collapses. FBBT lifts the lower bound
# off the invalid region first, so the solve succeeds at the true optimum
# (x = e, since the objective pushes x up against log(x) <= 1).
def build_log():
    m = pyo.ConcreteModel()
    m.x = pyo.Var(bounds=(-1e6, 1e6), initialize=-500.0)
    m.obj = pyo.Objective(expr=(m.x - 1000.0) ** 2)
    m.c = pyo.Constraint(expr=pyo.log(m.x) <= 1.0)
    return m


print("log(x) <= 1,  x in [-1e6, 1e6],  start x = -500   (true optimum: x = e = 2.71828)")
report(build_log)

log(x) <= 1,  x in [-1e6, 1e6],  start x = -500   (true optimum: x = e = 2.71828)
  FBBT off: Restoration Failed!                              iters=1  (no solution returned)
  FBBT on : Optimal Solution Found.                          iters=11  x*=2.718282
            Presolve FBBT: 2 sweeps, 1 variable tightenings (Σ|Δ|=1.000e6)


**FBBT off → `Restoration Failed!`** (1 iteration): the first evaluation
of `log(-500)` is NaN and the solver cannot recover. **FBBT on →
`Optimal Solution Found`** at `x = e ≈ 2.718`. A single bound
tightening — lifting the lower bound off the invalid `x ≤ 0` region — is
the whole difference between a crash and a clean solve.

### A wide box that fakes infeasibility

Same idea, a different failure mode. Minimize `(x − 3)²` subject to
`1/x ≥ 0.5` (i.e. `x ≤ 2`), but declare an enormous box `x ∈ [1e-9,
1e9]` and start at `x = 1e8`. From 100 million out the IPM wanders and
**reports the problem infeasible** — even though `x = 2` is a perfectly
good optimum. FBBT shrinks the box to the constraint geometry first, and
the solve lands on it.

In [7]:
# Example B — a box so wide the IPM gets lost. Minimize (x - 3)^2 subject to
# 1/x >= 0.5 (i.e. x <= 2), but declare x in [1e-9, 1e9] and start at x = 1e8.
# From 100 million out, the IPM wanders and declares the problem infeasible,
# even though x = 2 is a perfectly good optimum. FBBT collapses the box to the
# constraint geometry first, and the solve lands on it.
def build_recip():
    m = pyo.ConcreteModel()
    m.x = pyo.Var(bounds=(1e-9, 1e9), initialize=1e8)
    m.obj = pyo.Objective(expr=(m.x - 3.0) ** 2)
    m.c = pyo.Constraint(expr=1.0 / m.x >= 0.5)   # forces x <= 2
    return m


print("1/x >= 0.5,  x in [1e-9, 1e9],  start x = 1e8   (true optimum: x = 2)")
report(build_recip)

1/x >= 0.5,  x in [1e-9, 1e9],  start x = 1e8   (true optimum: x = 2)


  FBBT off: Converged to a point of local infeasibility. Problem may be infeasible. iters=4  (no solution returned)


  FBBT on : Optimal Solution Found.                          iters=13  x*=1.995013
            Presolve FBBT: 2 sweeps, 1 variable tightenings (Σ|Δ|=1.000e9)


## What the report fields mean

```
Presolve FBBT: 1 sweeps, 4 variable tightenings (Σ|Δ|=1.98e2)
```

* `sweeps` — number of outer iterations actually executed
  (≤ `fbbt_max_iter`, default 10).
* `variable tightenings` — total count of per-variable `(x_lo[j],
  x_hi[j])` updates that strictly improved the box.
* `Σ|Δ|` — sum of absolute bound improvements across all updates.

If FBBT detects infeasibility, you'll see an additional line:
`pounce: FBBT detected infeasibility (witness constraint N)`.

## When to turn FBBT on

FBBT is off by default (issue
[#62](https://github.com/jkitchin/pounce/issues/62)): on well-posed,
well-scaled problems it rarely changes the iteration count, and tighter
bounds can perturb the IPM's start, line search, and warm-start
behavior in either direction. The circle problem above is the typical
"wash" case — useful for seeing the mechanism, not for a speedup.

Its real value, shown by the two examples, is **robustness, not
iteration count**:

* **Domain safety.** When constraints use restricted-domain functions
  (`log`, `sqrt`, `1/x`, fractional powers) and the declared box reaches
  into the invalid region, the un-tightened box lets the IPM evaluate
  NaNs and collapse into restoration. FBBT removes the invalid region up
  front. *(`Restoration Failed` → `Optimal`.)*
* **Sane scaling on wide boxes.** A box spanning many orders of
  magnitude can strand the IPM in a region where it declares a feasible
  problem infeasible. FBBT shrinks the box to the constraint geometry
  first. *(false `infeasible` → `Optimal`.)*
* **Downstream presolve.** Tighter bounds enable extra row drops and can
  promote the LICQ verdict from `StructuralRank` to `Full`.

**Rule of thumb:** turn FBBT on when your model has loose or
order-of-magnitude-wide bounds — *especially* alongside
`log`/`sqrt`/division/fractional powers — or when the solver is failing
or stalling on a wide box. On tight, well-scaled models it is close to a
no-op, so leave it off until you have a reason.

## Tunable knobs

| Option | Default | Effect |
|---|---|---|
| `presolve_fbbt` | `no` | Master switch. Requires `presolve=yes`. |
| `fbbt_tol` | `1e-6` | Minimum per-variable bound improvement to keep iterating. |
| `fbbt_max_iter` | `10` | Outer-sweep cap. |
| `fbbt_max_constraints` | `0` | Per-sweep cap on constraints inspected (`0` = unlimited). |

## What FBBT can't see

Only TNLPs that expose constraint expression trees benefit from
FBBT. Today that's `.nl`-loaded problems via `NlTnlp` — which is
what Pyomo writes (and what this notebook used). Problems loaded
through `pounce.Problem` from Python class definitions or via the
C callback bridge currently opt out silently: the option is
accepted but produces no `Presolve FBBT:` line.

Adding FBBT support to another problem source means implementing
`pounce_nlp::expression_provider::ExpressionProvider` for it. See
the [FBBT reference](https://kitchingroup.cheme.cmu.edu/pounce/fbbt.html)
for the recipe.